In [ ]:
import pandas as pd

df = pd.read_csv("maternal_health_risk_mendeleydataset.csv")
df.head()

In [ ]:
import pandas as pd
import numpy as np

# 1. Dictionary covering absolutely every Bangla column in your printout
complete_translation_map = {
    'গর্ভস্হ শিশু অবস্থান': 'Fetal_Position',
    'গর্ভস্হ শিশু নাড়াচাড়া': 'Fetal_Movement',
    'গর্ভস্হ শিশু হৃৎস্পন্দন': 'Fetal_Heart_Rate',
    'প্রসাব পরিক্ষা এলবুমিন': 'Urine_Albumin',
    'প্রসাব পরিক্ষা সুগার': 'Urine_Sugar',
    'ঝুকিপূর্ণ গর্ভ': 'High_Risk_Pregnancy'
}

# 2. Rename the remaining columns
df.rename(columns=complete_translation_map, inplace=True)

# 3. Clean 'Fetal_Heart_Rate' (removes the 'm' from 140m -> 140)
df['Fetal_Heart_Rate'] = df['Fetal_Heart_Rate'].astype(str).str.replace('m', '', case=False)
df['Fetal_Heart_Rate'] = pd.to_numeric(df['Fetal_Heart_Rate'], errors='coerce')

# 4. Split Blood_Pressure ('100/60') into two numeric columns
if 'Blood_Pressure' in df.columns:
    df[['Systolic_BP', 'Diastolic_BP']] = df['Blood_Pressure'].str.split('/', expand=True).astype(float)
    df.drop(columns=['Blood_Pressure'], inplace=True)

# Let's see your final, perfectly clean columns
print(df.columns.tolist())
print(df.head())


In [ ]:
df.head(5)

In [ ]:

# drop the identifier
df = df.drop(columns=["Name"])

# "1st", "2nd" -> 1, 2
for col in ["Gravida", "Tetanus_Vaccine"]:
    df[col] = df[col].astype(str).str.extract(r"(\d+)").astype(float)

# 5.3'' -> 5 ft 3 in -> cm
def to_cm(h):
    if pd.isna(h):
        return None
    ft, inch = str(h).replace("''", "").split(".")
    return (int(ft) * 12 + int(inch)) * 2.54

df["Height_CM"] = df["Height_Feet_Inches"].apply(to_cm)
df = df.drop(columns=["Height_Feet_Inches"])

print(df.isna().sum())
print(df["High_Risk_Pregnancy"].value_counts())

In [ ]:
for col in ["Anemia", "Jaundice", "Urine_Albumin", "Urine_Sugar", "VDRL", "HRsAG"]:
    print(col, df[col].value_counts(dropna=False).to_dict())

In [ ]:
sev = {"None": 0, "Minimal": 1, "Medium": 2, "Higher": 3}
for col in ["Anemia", "Urine_Albumin"]:
    df[col] = df[col].fillna("None").map(sev)

df["has_jaundice"] = df["Jaundice"].notna().astype(int)
df = df.drop(columns=["Jaundice"])

In [ ]:
print(pd.crosstab(df["VDRL"], df["High_Risk_Pregnancy"], normalize="index"))
print(pd.crosstab(df["Urine_Sugar"], df["High_Risk_Pregnancy"], normalize="index"))

In [ ]:
df = df.drop(columns=["VDRL", "Urine_Sugar"])

In [ ]:
num_cols = ["Age", "Gravida", "Gestational_Age_Weeks", "Weight_KG",
            "Systolic_BP", "Diastolic_BP", "Height_CM", "Fetal_Heart_Rate"]
print(df.groupby("High_Risk_Pregnancy")[num_cols].mean().round(1).T)

In [ ]:
print(df.duplicated().sum())
print(df.drop(columns=["High_Risk_Pregnancy"]).duplicated().sum())

In [ ]:
for col in ["Anemia", "Urine_Albumin", "Fetal_Position", "Fetal_Movement", "HRsAG", "Tetanus_Vaccine", "has_jaundice"]:
    print(pd.crosstab(df[col], df["High_Risk_Pregnancy"], normalize="index").round(3), "\n")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# Model A: all columns
X_all = pd.get_dummies(df.drop(columns=["High_Risk_Pregnancy"]), drop_first=True)
# Model B: only useful columns
X_few = pd.get_dummies(df[["Tetanus_Vaccine", "HRsAG", "Weight_KG", "Systolic_BP"]], drop_first=True)
y = (df["High_Risk_Pregnancy"] == "Yes").astype(int)

for name, X in [("All columns", X_all), ("Few columns", X_few)]:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    m = RandomForestClassifier(random_state=42).fit(Xtr, ytr)
    print(name)
    print(classification_report(yte, m.predict(Xte)))

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X_all, y, test_size=0.2, stratify=y, random_state=42)
m = RandomForestClassifier(random_state=42).fit(Xtr, ytr)

imp = pd.Series(m.feature_importances_, index=X_all.columns).sort_values(ascending=False)
print(imp.head(10).round(3))

In [ ]:
imp.head(10).sort_values().plot(kind="barh", title="Top 10 feature importance")

In [ ]:
import joblib

joblib.dump(m, "maternal_model.pkl")
joblib.dump(list(X_all.columns), "model_columns.pkl")

In [ ]:
import joblib

model = joblib.load("maternal_model.pkl")
cols = joblib.load("model_columns.pkl")

# take 1 mother from the test set
sample = Xte.iloc[[0]][cols]
print("Model says:", model.predict(sample)[0])   # 1 = high risk, 0 = not
print("Real answer:", yte.iloc[0])

In [ ]:
import os
print(os.getcwd())

In [ ]:
import os
print(os.getcwd())
print([f for f in os.listdir() if f.endswith((".pkl", ".ipynb", ".csv"))])

In [ ]:
print(list(X_all.columns))